# p_1 — 04 · the overnight run

Trains **Config B** (n_layer 4 · d_model 256 · ctx 512 · BPE vocab 4096 · **4,339,200 params**)
on the health Q&A corpus with our own AdamW, cosine schedule, gradient clipping and
**atomic 15-minute checkpoints to Drive**.

**Sessions WILL die — that's normal weather.** Every ~15 min the trainer saves
`ckpt_stepN.npz` + `latest.txt` to `MyDrive/p_1/checkpoints/`; on restart the notebook
auto-resumes from the newest checkpoint (data batches are replayed deterministically
per step, so the loss curve continues without a spike).

Measured expectations (dev sandbox, CPU NumPy — Colab CPU is the same order):
- corpus = 464,588 train BPE tokens (4,064 entries) · ~4.3 chars/token
- batch B=4 × ctx 512 = 2,048 tok/step at ~2.5–2.9 s/step → **1 epoch ≈ 226 steps**
- defaults below: 6,000 steps ≈ **26 epochs ≈ 4.5–5 h** (fits one overnight session; it resumes if it doesn't)
- B=4 peaks ~2.1 GB RAM. If Colab shows >8 GB RAM you can raise `BATCH_SIZE` to 8 (≈2× speed per token).

## 0 · Environment — Drive, paths, repo

In [ ]:
import os, sys, pathlib
IN_COLAB = 'google.colab' in str(getattr(__import__('sys'), 'modules', {}) or '')
try:
    from google.colab import drive  # noqa
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    pass

if IN_COLAB:
    ROOT = pathlib.Path('/content/drive/MyDrive/p_1')
    # training code lives in the cloned repo (run 00_git_setup.ipynb first)
    REPO = pathlib.Path('/content/p_1')
else:
    ROOT = pathlib.Path('.').resolve()
    REPO = ROOT

sys.path.insert(0, str(REPO / 'colab'))
for sub in ('corpus', 'checkpoints', 'results'):
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
print('IN_COLAB =', IN_COLAB, '| ROOT =', ROOT, '| REPO =', REPO)

## 1 · Data + tokenizer (our BPE, merges.json from Task 3)

In [ ]:
from tokenizer_py import P1BPE
from train_lib import (AdamW, CosineWarmup, Trainer, build_data, plot_history,
                       load_health_qa, find_resume, resume_provenance)

CORPUS   = ROOT / 'corpus' / 'health_qa.jsonl'
MERGES   = ROOT / 'corpus' / 'merges.json'
CKPT_DIR = ROOT / 'checkpoints'

tok = P1BPE(str(MERGES))
data = build_data(str(CORPUS), tok, ctx=512, seed=1234, max_val_windows=32)
st = data.stats()
print(f"train entries {st['n_train_entries']:,} -> {st['train_tokens']:,} BPE tokens")
print(f"val   entries {st['n_val_entries']:,} -> {st['val_tokens']:,} BPE tokens")
print(f"1 epoch ~ {st['train_tokens'] // (4 * 512)} steps at B=4, ctx=512")

## 2 · Model + optimizer — **auto-resume happens here**

`Trainer` checks `MyDrive/p_1/checkpoints/` and, if a checkpoint exists (any prior
session), restores params + AdamW moments + step count and continues from there.
A fresh start prints `resume: none — fresh start`.

In [ ]:
import importlib
M03 = importlib.import_module('03_model')

CONFIG_B = dict(M03.CONFIG_B)          # n_layer 4 · d_model 256 · ctx 512 · vocab 4096
SEED     = 42
BATCH_SIZE = 4                         # raise to 8 if Colab RAM allows

model = M03.Transformer(CONFIG_B, seed=SEED)
print('params:', f"{model.numParams():,}")

import tokenizer_py
tokver = 'p1-bpe-v1:3838'              # format + merge count of corpus/merges.json
trainer = Trainer(model, tokver, str(CKPT_DIR), data,
                  cfg={'lr': 3e-4, 'warmup': 200, 'total_steps': 6000,
                       'batch_size': BATCH_SIZE, 'clip': 1.0,
                       'eval_every': 250, 'log_every': 50,
                       'ckpt_every_minutes': 15.0, 'seed': SEED})
print(trainer.resume_info or 'resume: none — fresh start')

## 3 · Train (safe to interrupt — really)

Stop the cell, close the tab, let Colab die — the next run of cell 2 + 3 continues
from the last 15-minute checkpoint. `max_minutes` caps the session; a clean finish
or a timeout both leave a final checkpoint.

In [ ]:
history = trainer.train(max_minutes=480)   # overnight: ~8 h ceiling, ends earlier at 6,000 steps
plot_history(history)                      # matplotlib = pencil, not intelligence
print('final val loss:', [r for r in history if r['kind'] == 'val'][-1])

## 4 · Listen to it speak (temperature sampling from the trained brain)

In [ ]:
import numpy as np

def generate(prompt='Q:', n=150, temperature=0.8, seed=None):
    rng = np.random.default_rng(seed)
    ids = [tok.bos] + tok.encode(prompt)
    ids = ids[-CONFIG_B['ctx'] + 1:]
    for _ in range(n):
        window = np.array(ids[-CONFIG_B['ctx']:])
        logits = model.forward(window, 1)              # [T, vocab]
        last = logits.data.reshape(logits.shape[-2:]).astype(np.float64)[-1]
        last /= temperature
        last -= last.max()
        p = np.exp(last); p /= p.sum()
        nxt = int(rng.choice(len(p), p=p))
        if nxt == tok.eos:
            break
        ids.append(nxt)
    return tok.decode(ids[1:])

for s in (42, 7, 1234):
    print('---', generate('Q:', n=180, temperature=0.8, seed=s))

### DONE WHEN
- [ ] An overnight run shows **smoothly falling val loss** (cell 3's curve; CSV in `checkpoints/train_log.csv`).
- [ ] **Kill test**: ≥15 min after the first checkpoint, `Runtime → Interrupt execution` (or just let the session die), re-run cells 0→3 — the trainer resumes and the loss curve shows **no spike** at the resume step.

*(The machinery is already proven locally: `scripts/test_task5_train.py` reproduced a
kill-at-75/resume-to-150 run whose losses were **bit-identical** to uninterrupted training.)*